### Instalações dos pacotes de base

In [1]:
%pip install python-dotenv
%pip install -U langgraph langchain langchain-google-genai langchain-tavily python-dotenv aiosqlite

Note: you may need to restart the kernel to use updated packages.
  Attempting uninstall: langgraph━━━━━━━━━━━━━━━ 0/3 [aiosqlite]
    Found existing installation: langgraph 1.1.3 0/3 [aiosqlite]
    Uninstalling langgraph-1.1.3:━━━━━━━━━━━ 0/3 [aiosqlite]
      Successfully uninstalled langgraph-1.1.3━━━━━━━━━━━━━━━━━━━━ 1/3 [langgraph]
  Attempting uninstall: langchain0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [langgraph]
    Found existing installation: langchain 1.2.13━━━━━━━━━━━━━ 1/3 [langgraph]
    Uninstalling langchain-1.2.13:0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [langgraph]
      Successfully uninstalled langchain-1.2.13━━━━━━━━━━━━━━━ 1/3 [langgraph]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain]/3 [langchain]
Note: you may need to restart the kernel to use updated packages.


### Importações das chaves do ambiente

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

### Instalação do pacote para checkpointer

In [3]:
%pip install langgraph-checkpoint-sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langgraph-checkpoint-sqlite]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import operator
from typing import Annotated, List, Any, Dict
from dataclasses import dataclass, field
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, BaseMessage, AnyMessage

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.checkpoint.sqlite import SqliteSaver

from typing_extensions import TypedDict

import sqlite3

### Define o estado do agente e a persistência da memória com SQLite

In [5]:
# Estado do agente
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# Persistência da sua memória
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        
        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_gemini)

        graph.add_node("action", self.take_action)

        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")
        '''
        Define o nó de início do grafo.
        '''

        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_gemini(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        print("Mensagens enviadas ao modelo:", messages)
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        '''
        Verifica se a última mensagem do modelo é uma chamada de ferramenta
        '''
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        '''
        Executa as ferramentas chamadas pelo modelo
        '''
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling Tool: {t['name']} with args: {t['args']}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Returning to LLM after action!")
        return {'messages': results}

###

## Definição de uma estrutura para início de um chat
- tool
- prompt_system
- model
- abot (agent)

In [12]:
current_tavily_api_key = os.getenv('TAVILY_API_KEY')
if not current_tavily_api_key:
    raise ValueError("TAVILY_API_KEY não encontrada. Certifique-se de que está no seu .env e python-dotenv está instalado.")

tool = TavilySearch(max_results=3, tavily_api_key=current_tavily_api_key)


prompt_system = """Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.
Você tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).
Busque informações apenas quando tiver certeza do que procurar.
Se precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.
Quando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas."""

model = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

abot = Agent(model, [tool], system=prompt_system, checkpointer=memory)
'''
Objeto instanciado com atributo "checkpointer" para persistência de memória
'''

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


'\nObjeto instanciado com atributo "checkpointer" para persistência de memória\n'

### Definição de uma interação, aplicando a ideia de contexto através da chave thread_id

In [16]:
messages = [HumanMessage(content="Como estava o tempo em São Paulo em 21/03/2026?")]
thread = {"configurable": {"thread_id": "1"}}
'''
A chave thread_id define o contexto do chat
'''


print("\n--- Pergunta 1: Tempo em São Paulo ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"): 
             print(f"{k}: {v['messages']}")


--- Pergunta 1: Tempo em São Paulo ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': 'Não consigo prever 

#### Usando o mesmo thread_id para manter o contexto de chat

In [14]:
messages = [HumanMessage(content="E no Rio de Janeiro?")]
thread = {"configurable": {"thread_id": "1"}} 

print("\n--- Pergunta 2: Tempo no Rio de Janeiro ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 2: Tempo no Rio de Janeiro ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': 'Não consigo pr

In [15]:
messages = [HumanMessage(content="Qual está mais quente?")]
thread = {"configurable": {"thread_id": "1"}} 

print("\n--- Pergunta 3: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 3: Comparação ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': 'Não consigo prever o tempo 

In [18]:
messages = [HumanMessage(content="Qual é a do seu sistema hoje?")]
thread = {"configurable": {"thread_id": "1"}}

print("\n--- Pergunta 3: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 3: Comparação ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Como está o tempo em São Paulo hoje (21/08/2025)?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'type': 'text', 'text': 'Não consigo prever o tempo 

### Trocando o contexto do chat, alterandoa chave thread_id, simulando um novo chat ou usuário.

In [19]:
messages = [HumanMessage(content="Qual está mais quente?")]
thread = {"configurable": {"thread_id": "2"}} 

print("\n--- Pergunta 4: Comparação ---")
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action"):
            print(f"{k}: {v['messages']}")


--- Pergunta 4: Comparação ---
Mensagens enviadas ao modelo: [SystemMessage(content='Você é um assistente de pesquisa inteligente. Use o mecanismo de busca (tavily_search_results_json) para procurar informações.\nVocê tem permissão para fazer múltiplas chamadas à ferramenta (em conjunto ou em sequência).\nBusque informações apenas quando tiver certeza do que procurar.\nSe precisar de mais detalhes para formular uma pergunta de acompanhamento, você tem permissão para fazer isso.\nQuando solicitado a comparar informações (ex: qual é mais quente, maior, etc.), use as informações do histórico da conversa e dos resultados das ferramentas.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Qual está mais quente?', additional_kwargs={}, response_metadata={})]
llm: [AIMessage(content=[{'type': 'text', 'text': 'O que você gostaria de comparar?', 'extras': {'signature': 'Co8GAb4+9vsgMQixXohgWWmxUqXRb0CmGegyaNgQA41bjm3MEZf3eGQRb2uBJSzJS24e8ib0EiafBb/twy+eI4DtLbAwPn/um9s+F8T/s2E